# RQ1 robustness tests

Checks reported in Section 4.4: SOC3 fixed effects, individual-occupation
exclusions, GPT-4 Beta, core-weighted shares, and median wage.

Set `BASE_DIR` to the project folder containing the three frozen input CSVs,
then run the notebook from top to bottom. The notebook displays results and
retains unrounded result tables in memory; it does not export files.

M0 contains aggregate exposure; M1 adds mechanism composition; M2 adds
structural states and conditional importance positions. All fits use OLS with
HC3 covariance, normal-based coefficient intervals, and Wald F joint tests.
The input outcomes are annualized log changes. Coefficients, standard errors,
and interval bounds are displayed on the manuscript's scale (×100).

## 1. Inputs and common samples

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from IPython.display import display
from statsmodels.stats.outliers_influence import OLSInfluence

BASE_DIR = Path("/content/drive/MyDrive/GenAI-task-affordance-labor-outcomes")
INPUTS = {
    "configuration": "05_Layer3_Task_Configuration/Output/Occupational_Task_Affordance_Configuration_v7.csv",
    "exposure": "07_GenAI_Exposure/Output/Occupational_Configuration_Exposure_v1.csv",
    "outcomes": "08_Labor_Outcomes/Output/OEWS_Occupational_Outcomes_v1.csv",
}
MECHANISMS = ["automation", "augmentation", "limited"]
SHARES = ["automation_share", "augmentation_share"]
WEIGHTED_SHARES = ["automation_core_weighted_share", "augmentation_core_weighted_share"]
STATES = ["automation_present", "augmentation_absent", "limited_absent"]
POSITIONS = [f"{m}_position_centered" for m in MECHANISMS]
OUTCOMES = {
    "Employment": ("employment_ann_log_change", 540),
    "Wage": ("mean_wage_ann_log_change", 537),
    "Median wage": ("median_wage_ann_log_change", 526),
}
MAIN_OUTCOMES = ["Employment", "Wage"]

In [ ]:
inputs = {}
for name, path in INPUTS.items():
    path = BASE_DIR / path
    if not path.is_file():
        raise FileNotFoundError(f"Missing frozen input: {path}. Set BASE_DIR before running.")
    frame = pd.read_csv(path, dtype={"soc6": "string"})
    frame["soc6"] = frame["soc6"].str.strip()
    assert frame["soc6"].notna().all() and frame["soc6"].is_unique, name
    assert frame["soc6"].str.fullmatch(r"\d{2}-\d{4}").all(), name
    inputs[name] = frame

data = inputs["configuration"][["soc6", *[
    f"{m}_{field}" for m in MECHANISMS
    for field in ["count", "share", "position", "core_weighted_share"]
]]].merge(
    inputs["exposure"][["soc6", "human_beta", "gpt4_beta"]],
    on="soc6", how="left", sort=False, validate="one_to_one",
).merge(
    inputs["outcomes"][["soc6", *[column for column, _ in OUTCOMES.values()]]],
    on="soc6", how="left", sort=False, validate="one_to_one",
)
assert len(data) == 604 and data["soc6"].is_unique

# A missing position excludes a present category, but not an absent category.
eligible = pd.Series(True, index=data.index)
for mechanism in MECHANISMS:
    present = data[f"{mechanism}_count"].gt(0)
    eligible &= ~(present & data[f"{mechanism}_position"].isna())
    data[f"{mechanism}_position_centered"] = (
        data[f"{mechanism}_position"] - 0.5
    ).where(present, 0.0)

data["automation_present"] = data["automation_count"].gt(0).astype("int8")
data["augmentation_absent"] = data["augmentation_count"].eq(0).astype("int8")
data["limited_absent"] = data["limited_count"].eq(0).astype("int8")
data["soc3"] = (data["soc6"].str[:2] + "-" + data["soc6"].str[3]).astype(object)
eligible &= data[["human_beta", *[f"{m}_share" for m in MECHANISMS]]].notna().all(axis=1)

samples = {}
for outcome, (column, expected_n) in OUTCOMES.items():
    samples[outcome] = data.loc[eligible & data[column].notna()].copy()
    assert len(samples[outcome]) == expected_n, f"Unexpected {outcome} sample size."

display(pd.DataFrame({"Outcome": list(OUTCOMES), "N": [len(samples[o]) for o in OUTCOMES]}))

## 2. Shared model and reporting functions

In [ ]:
def fit_models(frame, outcome, exposure="human_beta", shares=SHARES, soc3=False):
    terms = {
        "M0": [exposure],
        "M1": [exposure, *shares],
        "M2": [exposure, *shares, *STATES, *POSITIONS],
    }
    column = OUTCOMES[outcome][0]
    assert np.isfinite(frame[[column, *terms["M2"]]].to_numpy(dtype=float)).all()
    models = {}
    for name, predictors in terms.items():
        if soc3:
            model = smf.ols(f"{column} ~ {' + '.join(predictors)} + C(soc3)",
                            data=frame, missing="raise")
        else:
            model = sm.OLS(frame[column], sm.add_constant(frame[predictors],
                           has_constant="add"), missing="raise")
        assert np.linalg.matrix_rank(model.exog) == model.exog.shape[1], name
        result = model.fit(cov_type="HC3", use_t=False)
        assert result.nobs == len(frame)
        assert pd.Index(result.model.data.row_labels).equals(frame.index)
        assert result.get_influence().hat_matrix_diag.max() < 1 - 1e-8
        assert np.isfinite(result.cov_params().to_numpy()).all()
        models[name] = result
    return models


def joint_tests(models, outcome, specification, shares=SHARES):
    blocks = [
        ("Mechanism composition", "M1", "M1 vs. M0", shares),
        ("Structural states and importance positions", "M2", "M2 vs. M1", STATES + POSITIONS),
        ("Full configuration", "M2", "M2 vs. M0", shares + STATES + POSITIONS),
        ("Importance positions", "M2", "Within M2", POSITIONS),
    ]
    rows = []
    for block, name, comparison, terms in blocks:
        model = models[name]
        restriction = np.eye(len(model.params))[[model.params.index.get_loc(t) for t in terms]]
        test = model.wald_test(restriction, use_f=True, scalar=True)
        assert int(test.df_num) == len(terms)
        assert np.isfinite([test.statistic, test.pvalue]).all()
        rows.append({
            "Outcome": outcome, "Specification": specification, "N": int(model.nobs),
            "Block": block, "Comparison": comparison, "Model": name,
            "df_num": int(test.df_num), "df_denom": int(test.df_denom),
            "F": float(test.statistic), "p": float(test.pvalue),
        })
    return pd.DataFrame(rows)


def coefficients(model, terms):
    intervals = model.conf_int().loc[terms] * 100
    return pd.DataFrame({
        "Term": terms, "Estimate": model.params.loc[terms].to_numpy() * 100,
        "HC3 SE": model.bse.loc[terms].to_numpy() * 100,
        "CI lower": intervals.iloc[:, 0].to_numpy(),
        "CI upper": intervals.iloc[:, 1].to_numpy(),
        "p": model.pvalues.loc[terms].to_numpy(),
    })


def format_p(value):
    return "< .001" if value < .001 else f"{value:.4f}".lstrip("0")


def display_tests(results):
    for outcome, part in results.groupby("Outcome", sort=False):
        table = part.assign(
            Column=part["Specification"] + " (N=" + part["N"].astype(str) + ")",
            Result=part.apply(
                lambda r: f"F({r.df_num}, {r.df_denom}) = {r.F:.3f}; "
                          f"p {format_p(r.p) if r.p < .001 else '= ' + format_p(r.p)}",
                axis=1,
            ),
        )
        print(outcome)
        display(table.pivot(index=["Block", "Comparison"], columns="Column", values="Result").reindex(
            index=pd.MultiIndex.from_frame(part[["Block", "Comparison"]].drop_duplicates()),
            columns=table["Column"].unique(),
        ))

In [ ]:
# Fit the reference models once; reuse them in every comparison.
primary_models = {outcome: fit_models(frame, outcome) for outcome, frame in samples.items()}

## 3. SOC3 fixed effects

In [ ]:
soc3_models, soc3_tests, soc3_coefficients, singleton_rows = {}, [], [], []
for outcome in MAIN_OUTCOMES:
    frame = samples[outcome]
    keep = frame.groupby("soc3")["soc3"].transform("size").ge(2)
    singleton_rows.append(frame.loc[~keep, ["soc6", "soc3"]].assign(Outcome=outcome))
    matched = frame.loc[keep]
    assert len(matched) == {"Employment": 535, "Wage": 532}[outcome]
    soc3_models[outcome] = {
        "Primary": primary_models[outcome],
        "Matched sample": fit_models(matched, outcome),
        "SOC3 FE": fit_models(matched, outcome, soc3=True),
    }
    for specification, models in soc3_models[outcome].items():
        soc3_tests.append(joint_tests(models, outcome, specification))
        soc3_coefficients.append(coefficients(models["M2"], ["human_beta", *SHARES, *STATES, *POSITIONS]).assign(
            Outcome=outcome, Specification=specification, N=int(models["M2"].nobs)))

singletons = pd.concat(singleton_rows, ignore_index=True)
soc3_tests = pd.concat(soc3_tests, ignore_index=True)
soc3_coefficients = pd.concat(soc3_coefficients, ignore_index=True)
print("Excluded singleton occupations")
display(singletons[["Outcome", "soc6", "soc3"]])
for outcome in MAIN_OUTCOMES:
    table = soc3_coefficients.loc[soc3_coefficients["Outcome"].eq(outcome)].copy()
    table["Estimate [95% CI]"] = table.apply(
        lambda r: f"{r['Estimate']:.4f} [{r['CI lower']:.4f}, {r['CI upper']:.4f}]", axis=1)
    print(f"{outcome}: M2 coefficients on the annualized log change × 100 scale")
    display(table.pivot(index="Term", columns="Specification", values="Estimate [95% CI]").reindex(
        index=["human_beta", *SHARES, *STATES, *POSITIONS],
        columns=["Primary", "Matched sample", "SOC3 FE"],
    ))

## 4. Exposure benchmark

In [ ]:
benchmark_models, benchmark_tests = {}, []
for outcome in MAIN_OUTCOMES:
    frame = samples[outcome]
    assert frame["gpt4_beta"].notna().all() and frame["gpt4_beta"].between(0, 1).all()
    benchmark_models[outcome] = fit_models(frame, outcome, exposure="gpt4_beta")
    benchmark_tests.append(joint_tests(benchmark_models[outcome], outcome, "GPT-4 Beta"))

benchmark_tests = pd.concat(benchmark_tests, ignore_index=True)
table4_results = pd.concat([soc3_tests, benchmark_tests], ignore_index=True)
table4_results["Specification"] = pd.Categorical(
    table4_results["Specification"],
    categories=["Primary", "GPT-4 Beta", "Matched sample", "SOC3 FE"], ordered=True,
)
table4_results = table4_results.sort_values("Specification", kind="stable")
table4_results["Specification"] = table4_results["Specification"].astype(str)
print("Table 4. Joint tests by exposure measure and SOC3 FE")
display_tests(table4_results)

## 5. Individual-occupation exclusions

In [ ]:
def influential_occupations(models, frame):
    rows = []
    for name in ["M1", "M2"]:
        model = models[name]
        influence = OLSInfluence(model)
        leverage = influence.hat_matrix_diag
        cook = np.asarray(influence.cooks_distance[0])
        dfbetas = np.asarray(influence.dfbetas)

        assert np.all(leverage < 1 - 1e-8)
        assert np.isfinite(cook).all() and np.isfinite(dfbetas).all()

        selections = {
            "Cook's distance": pd.Series(cook).nlargest(5).index.tolist(),
            "Leverage": pd.Series(leverage).nlargest(3).index.tolist(),
        }
        for term in model.model.exog_names[2:]:
            j = model.model.exog_names.index(term)
            selections[f"DFBETAS: {term}"] = [
                int(np.abs(dfbetas[:, j]).argmax())
            ]

        for criterion, indices in selections.items():
            for i in indices:
                rows.append({
                    "Model": name,
                    "SOC6": str(frame.iloc[i]["soc6"]),
                    "Criterion": criterion,
                })
    return pd.DataFrame(rows)


selection_rows, omission_tests, omission_coefficients = [], [], []

for outcome in MAIN_OUTCOMES:
    frame = samples[outcome]
    selected = influential_occupations(primary_models[outcome], frame)
    selection_rows.append(selected.assign(Outcome=outcome))

    cases = sorted(selected["SOC6"].unique())
    assert set(cases).issubset(set(frame["soc6"]))

    print(f"{outcome}: {len(cases)} occupations; each excluded separately")
    print(", ".join(cases))

    for case in ["Full sample", *cases]:
        if case == "Full sample":
            models = primary_models[outcome]
        else:
            reduced = frame.loc[frame["soc6"].ne(case)]
            assert len(reduced) == len(frame) - 1
            models = fit_models(reduced, outcome)

        omission_tests.append(joint_tests(models, outcome, case))

        if outcome == "Employment":
            for name in ["M1", "M2"]:
                omission_coefficients.append(
                    coefficients(models[name], SHARES).assign(
                        Excluded_SOC6=case,
                        Model=name,
                        N=int(models[name].nobs),
                    )
                )

influence_selection = pd.concat(selection_rows, ignore_index=True)
omission_tests = pd.concat(
    omission_tests, ignore_index=True
).rename(columns={"Specification": "Excluded SOC6"})
omission_coefficients = pd.concat(
    omission_coefficients, ignore_index=True
)

summary = []
for (outcome, block), part in omission_tests.groupby(
    ["Outcome", "Block"], sort=False
):
    excluded = part.loc[part["Excluded SOC6"].ne("Full sample"), "p"]
    summary.append({
        "Outcome": outcome,
        "Block": block,
        "Full-sample p": part.loc[
            part["Excluded SOC6"].eq("Full sample"), "p"
        ].iloc[0],
        "Minimum p": excluded.min(),
        "Maximum p": excluded.max(),
    })

table5_results = pd.DataFrame(summary)
print("Table 5. Joint tests after individual occupation exclusion")
display(table5_results.style.hide(axis="index").format({
    "Full-sample p": format_p,
    "Minimum p": format_p,
    "Maximum p": format_p,
}))

## 6. Core-weighted shares

Replace the two unweighted composition regressors with their frozen
core-weighted shares. Retain each outcome's original sample, Human Beta,
structural states, and importance positions. This changes the task-share measure;
the regressions remain unweighted OLS.

In [ ]:
weighted_models, weighted_tests, weighted_coefficients = {}, [], []
for outcome in MAIN_OUTCOMES:
    frame = samples[outcome]
    shares = frame[[f"{m}_core_weighted_share" for m in MECHANISMS]]
    assert np.isfinite(shares.to_numpy()).all()
    assert ((shares >= 0) & (shares <= 1)).all().all()
    assert np.allclose(shares.sum(axis=1), 1, rtol=0, atol=1e-10)
    weighted_models[outcome] = fit_models(frame, outcome, shares=WEIGHTED_SHARES)
    weighted_tests.extend([
        joint_tests(primary_models[outcome], outcome, "Unweighted shares"),
        joint_tests(weighted_models[outcome], outcome, "Core-weighted shares", WEIGHTED_SHARES),
    ])
    weighted_coefficients.append(coefficients(weighted_models[outcome]["M2"],
        ["human_beta", *WEIGHTED_SHARES, *STATES, *POSITIONS]).assign(Outcome=outcome))

weighted_tests = pd.concat(weighted_tests, ignore_index=True)
weighted_coefficients = pd.concat(weighted_coefficients, ignore_index=True)
display_tests(weighted_tests)

## 7. Median wage

Use median wage instead of mean wage with the same M0–M2 specifications and
Human Beta. Each outcome uses its own common sample (mean wage: 537; median
wage: 526), as in the original analysis.

In [ ]:
median_tests = joint_tests(primary_models["Median wage"], "Median wage", "Human Beta")
median_coefficients = coefficients(primary_models["Median wage"]["M2"],
                                   ["human_beta", *SHARES, *STATES, *POSITIONS])
display_tests(median_tests)